# Lab 1 — THE LINE
**هندسة تطبيقات النماذج اللغوية الكبيرة**  
**إعداد: أ. لطيفة سعد**

## ① DEMO v0 — النسخة الأولى

In [1]:
def ask_the_line(question):
    return f"THE LINE received: {question}"

In [2]:
print(
    ask_the_line(
        "How can we reduce cooling energy in a sustainable building near Tabuk?"
    )
)

THE LINE received: How can we reduce cooling energy in a sustainable building near Tabuk?


## ② LLMRequest + LLMResponse

In [3]:
from dataclasses import dataclass, field
from typing import Protocol

@dataclass
class LLMRequest:
    messages: list[dict]
    model: str
    max_tokens: int = 512

@dataclass
class LLMResponse:
    text: str

## ③ LLMClient — الـInterface

In [4]:
class LLMClient(Protocol):
    def complete(
        self,
        request: LLMRequest
    ) -> LLMResponse:
        ...

## ④ OpenAI-Compatible Adapter

In [5]:
class OpenAICompatClient:
    def complete(
        self,
        request: LLMRequest
    ) -> LLMResponse:
        question = request.messages[-1]["content"]
        return LLMResponse(
            text=f"THE LINE response to: {question}"
        )

## ⑤ FIRST REQUEST — English

In [6]:
the_line = OpenAICompatClient()

request = LLMRequest(
    messages=[
        {
            "role": "user",
            "content": "How can we reduce cooling energy in a sustainable building near Tabuk?"
        }
    ],
    model="demo-model"
)

response = the_line.complete(request)
print(response.text)

THE LINE response to: How can we reduce cooling energy in a sustainable building near Tabuk?


## ⑥ FIRST REQUEST — Arabic

In [7]:
request = LLMRequest(
    messages=[
        {
            "role": "user",
            "content": "كيف يمكن تقليل استهلاك التكييف في مبنى مستدام قريب من تبوك؟"
        }
    ],
    model="demo-model"
)

response = the_line.complete(request)
print(response.text)

THE LINE response to: كيف يمكن تقليل استهلاك التكييف في مبنى مستدام قريب من تبوك؟


## ⑦ CONVERSATION STATE

In [8]:
@dataclass
class ConversationState:
    max_turns: int = 8
    messages: list[dict] = field(default_factory=list)

    def add(self, role, content):
        self.messages.append({
            "role": role,
            "content": content
        })
        self.messages = self.messages[
            -(self.max_turns * 2):
        ]

    def get_messages(self):
        return self.messages

## ⑧ TEST — Conversation Memory

In [9]:
conversation = ConversationState(max_turns=8)

conversation.add(
    "user",
    "I am an engineer from Tabuk."
)

conversation.add(
    "assistant",
    "Nice to meet you."
)

conversation.add(
    "user",
    "What areas can you help me with?"
)

conversation.add(
    "assistant",
    "I can help with energy, water, materials, and sustainable design questions."
)

print(conversation.get_messages())

[{'role': 'user', 'content': 'I am an engineer from Tabuk.'}, {'role': 'assistant', 'content': 'Nice to meet you.'}, {'role': 'user', 'content': 'What areas can you help me with?'}, {'role': 'assistant', 'content': 'I can help with energy, water, materials, and sustainable design questions.'}]


## ⑨ MAKE CHAT

In [10]:
def make_chat(conversation, client, user_message):
    conversation.add(
        "user",
        user_message
    )
    request = LLMRequest(
        messages=conversation.get_messages(),
        model="demo-model"
    )
    response = client.complete(request)
    conversation.add(
        "assistant",
        response.text
    )
    return response.text

In [11]:
conversation = ConversationState(max_turns=8)
the_line = OpenAICompatClient()

In [12]:
print(
    make_chat(
        conversation,
        the_line,
        "I am an engineer from Tabuk."
    )
)

THE LINE response to: I am an engineer from Tabuk.


In [13]:
print(
    make_chat(
        conversation,
        the_line,
        "What areas can you help me with?"
    )
)

THE LINE response to: What areas can you help me with?


## ⑩ WINDOW = 8 TURNS

In [14]:
conversation = ConversationState(max_turns=8)

for i in range(1, 10):
    conversation.add(
        "user",
        f"Message {i}"
    )
    conversation.add(
        "assistant",
        f"Response {i}"
    )

print(conversation.get_messages())

[{'role': 'user', 'content': 'Message 2'}, {'role': 'assistant', 'content': 'Response 2'}, {'role': 'user', 'content': 'Message 3'}, {'role': 'assistant', 'content': 'Response 3'}, {'role': 'user', 'content': 'Message 4'}, {'role': 'assistant', 'content': 'Response 4'}, {'role': 'user', 'content': 'Message 5'}, {'role': 'assistant', 'content': 'Response 5'}, {'role': 'user', 'content': 'Message 6'}, {'role': 'assistant', 'content': 'Response 6'}, {'role': 'user', 'content': 'Message 7'}, {'role': 'assistant', 'content': 'Response 7'}, {'role': 'user', 'content': 'Message 8'}, {'role': 'assistant', 'content': 'Response 8'}, {'role': 'user', 'content': 'Message 9'}, {'role': 'assistant', 'content': 'Response 9'}]


## ⑪ DOCUMENTATION — Context Budget

In [15]:
from pathlib import Path

Path("docs").mkdir(exist_ok=True)

Path("docs/context_budget.md").write_text(
"""# THE LINE Context Budget
Total Context Budget: 16K tokens
- System Prompt: 2K
- Conversation History: 8K
- Tools: 2K
- Output: 4K

The conversation history is windowed to 8 turns.
""",
encoding="utf-8"
)

181